# Hyperparameter Tuning + Feature Engineering: Grid Search, Pipelines, and Leakage Detection

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2027S_predictive_analytics_QM474/main/notebooks/figures/mgmt_474_ai_logo_02-modified.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>QM47400 Predictive Analytics</center>
# <center>Professor: Davi Moreira </center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2027S_predictive_analytics_QM474/blob/main/notebooks/nb09_tuning_feature_engineering_project_baseline_student.ipynb)

---

> **🎯 nb09 = the toolkit closer.** By the end of today, you will hold the full mid-course toolkit — every concept (bias–variance, overfitting vs underfitting, leakage, regression vs classification, the curse of dimensionality), every sklearn primitive (`Pipeline`, `ColumnTransformer`, `Ridge`/`Lasso`/`LogisticRegression`, `cross_val_score`, `GridSearchCV`/`RandomizedSearchCV`), and every decision rule (`compare_paired` on identical folds read against a margin declared in advance, "a search's best cell is a nominee, not a champion", "fit inside the pipeline", "test set stays locked"). **Section C of today's notebook is your one-page consolidated recap** of everything nb01–nb09 has built — keep it close as a reference for any future analysis.

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Run `GridSearchCV` and `RandomizedSearchCV` on known models and read `cv_results_` as a table of nb08-style CV runs — five fold scores, their mean, and their SD per candidate
2. Treat a search's best cell as a **nominee** and decide whether it replaces the current setting with `compare_paired` — identical folds, the interval corrected for fold overlap, one of four outcomes against a margin declared in advance
3. Build a `ColumnTransformer` that handles both categorical and numeric features inside a single `Pipeline`
4. Use `FunctionTransformer` to embed domain-specific feature engineering (ratios, bins) inside the pipeline without leakage, and test the new feature with `compare_paired`
5. Detect a data-leakage bug in a provided pipeline by comparing CV scores before and after the fix
6. Explain why every feature-engineering step must live inside the pipeline that `cross_val_score` or `GridSearchCV` evaluates

---

> **📋 Participation Reminder:** This notebook contains **2 PAUSE-AND-DO exercises**, one at the end of each big section. Please complete them before submitting your notebook.

---

## 💼 Why This Matters: From One Knob to a Dial (and a Leaky Pipeline to Fix)

Two business cases are on your desk today — one returning from earlier notebooks, one brand new.

**Section A — HomeValue Analytics + MedScreen.** In nb05 you fit Ridge regression on California Housing across a few hand-picked $\alpha$ values — enough to feel how regularization strength trades off bias and variance, but stopping short of a *systematic* sweep on identical folds. nb08 then ran a clean head-to-head — Ridge ($\alpha = 1.0$) against plain OLS — and `compare_paired` found the two **practically equivalent**: the interval around Ridge's mean gain sat deep inside the CFO's $\pm 0.01$ $R^2$ margin, so OLS, the model HomeValue already runs, stayed. That verdict was clean *but* rested on one point of the regularization spectrum. The natural next question, the one the CFO would actually ask before signing off, is: *"Did OLS really hold up against every $\alpha$, or could $\alpha = 10$, $\alpha = 100$, or $\alpha = 0.01$ show a gain worth having?"* The tool that answers this is `GridSearchCV` — nb08's CV ritual run once per candidate $\alpha$ — and the discipline that keeps it honest is to treat its best cell as a *nominee* that still has to beat OLS with `compare_paired`. The Health Department wants the same treatment for MedScreen's logistic regression — a principled $C$-grid search instead of one hand-chosen value.

**Section B — TechCorp Talent Analytics.** You have been loaned to TechCorp, a 1,500-person SaaS company. The People Analytics team wants to flag employees at high risk of resigning in the next 6 months so HR can open retention conversations early. The economics are sharp: losing a mid-level engineer costs about USD 75,000 (recruitment + 6 months of productivity loss + ramp time); a retention conversation costs about USD 500 (manager time + modest retention bonus). Even a modest lift over "managers guessing" pays for itself many times over.

TechCorp's dataset has something California Housing and Breast Cancer never had: **real categorical columns** — `department`, `job_level`, `remote_status`, `manager_id`. That means you finally need `ColumnTransformer` to mix one-hot encoding with standardization in one pipeline. And while you were away, an intern shipped a first draft of the modeling pipeline. Their CV ROC-AUC looked suspiciously good… because it is. You will spot the leak, fix it, watch the inflated score collapse, and then find a second, subtler leak on your own.

By the end of today you own a workflow that plugs directly into your final project and the Course Case Competition: **ColumnTransformer → FunctionTransformer → GridSearchCV → the best cell as a nominee → `compare_paired` against the current setting → one final pipeline refit on all training data**. If that sentence feels abstract now, it will feel like a muscle memory by the end of the notebook.

> **Today's focus:** turn nb08's single CV run into a *table* of CV runs, learn that its top row is a nominee that must earn its place with `compare_paired`, and build the first pipeline in this course that handles real categorical data — all while staging two concrete leakage case studies so that "don't leak" stops being a lecture warning and becomes something you can spot in the wild.

---

In [ ]:
# Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.datasets import fetch_california_housing, load_breast_cancer
from sklearn.model_selection import (
    train_test_split, KFold, StratifiedKFold,
    cross_val_score, GridSearchCV, RandomizedSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge, LinearRegression, LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import roc_auc_score

import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

RANDOM_SEED = 474
np.random.seed(RANDOM_SEED)

# The course's one comparison tool (nb08), unchanged. A = challenger, B = current champion.
def compare_paired(scores_a, scores_b, margin):
    """Paired comparison on identical folds; higher score = better."""
    d = np.asarray(scores_a) - np.asarray(scores_b)
    k = len(d)
    t = stats.t.ppf(0.975, k - 1)
    se = d.std(ddof=1) * np.sqrt(1/k + 1/(k - 1))   # Nadeau–Bengio correction
    lo, hi = d.mean() - t*se, d.mean() + t*se
    if lo > margin:
        verdict = "A is better by a worthwhile amount"
    elif hi < -margin:
        verdict = "B is better by a worthwhile amount"
    elif -margin < lo and hi < margin:
        verdict = "Practically equivalent: choose the simpler model"
    else:
        verdict = "Unresolved: apply the default you declared"
    return d, d.mean(), (lo, hi), verdict

# Margins declared before any score is computed (the same numbers as nb08; base decides).
DELTAS_R2  = {'small': 0.005, 'base': 0.01, 'large': 0.02}   # HomeValue CFO
DELTAS_AUC = {'small': 0.005, 'base': 0.01, 'large': 0.02}   # MedScreen review board; TechCorp CHRO
DELTA_R2, DELTA_AUC = DELTAS_R2['base'], DELTAS_AUC['base']

def show_band(scores_a, scores_b, deltas, a_name, b_name):
    """Print the outcome at each declared margin and the champion it implies (A only on a worthwhile gain)."""
    champs = []
    for name, delta in deltas.items():
        _, _, _, v = compare_paired(scores_a, scores_b, delta)
        champ = a_name if v.startswith('A is better') else b_name
        champs.append(champ)
        print(f'  {name:>5} (δ = {delta:.3f}): {v:<50} → champion: {champ}')
    print('  → ' + ('Robust across the three declared margins (for these folds).' if len(set(champs)) == 1
                    else 'Margin-sensitive: the champion changes with the margin.'))

print('Setup complete.')
print(f'Random seed: {RANDOM_SEED}')

**Reading the output:**

The setup cell imports today's full toolbox in one shot, so nothing later has to stop and re-import. Here is a quick tour of what each piece does and where you will meet it.

The **two tuners** are `GridSearchCV` and `RandomizedSearchCV`. Both take a pipeline plus a parameter grid (or distribution), run 5-fold CV for every candidate configuration, and store every fold's score in a table called `cv_results_`. Think of `cv_results_` as "a stack of nb08 runs" — every row is one complete nb08 evaluation with its own five fold scores (`split0_test_score` … `split4_test_score`), mean, and SD. `GridSearchCV` enumerates every combination in a fixed grid; `RandomizedSearchCV` samples from distributions when the grid would be too big to enumerate.

**`ColumnTransformer`** is the sklearn idiom for *"different preprocessing per column type."* Today it lets you apply `StandardScaler` to the numeric columns and `OneHotEncoder(handle_unknown='ignore')` to the categorical columns inside a single `Pipeline`. `handle_unknown='ignore'` is a safety net: if a prediction-time row contains a category never seen in training, the encoder outputs zeros for that row's category block instead of crashing.

**`FunctionTransformer`** wraps a plain Python function — e.g., "compute a ratio column" — into a pipeline step so that the function refits correctly on each CV fold. You will use it in Section 2 for a domain feature that the HR Business Partner asks about.

**`SelectKBest`** picks the top-$k$ features by a univariate score (today, `f_classif`). It is genuinely useful — *and* it is the transform you will misuse on purpose in PAUSE-AND-DO 2 to see how leakage feels when you accidentally set it up wrong.

**`compare_paired`** is nb08's comparison, pasted unchanged: it takes the fold scores of a challenger (A) and of the current champion (B), computed on the *same* folds, and returns the per-fold gains, their mean, the interval corrected for fold overlap, and one of four outcomes. **`scipy.stats`** supplies the Student's $t$ value inside it — and `loguniform` for the random search in §1.2. The **margins** (`DELTAS_R2`, `DELTAS_AUC`) are the same ones nb08 declared, written down again before any score exists; the base margin, 0.01, decides, and `show_band` prints the outcome at all three so you can see whether a decision depends on the margin.

`RANDOM_SEED = 474` is inherited from nb01–nb08, so every fold is directly comparable to what you computed in prior notebooks.

> **A question that often comes up here:** *"Do I need to know exactly when to reach for `GridSearchCV` versus `RandomizedSearchCV`?"* Simple rule: if the grid is under \~100 combinations, use `GridSearchCV` — exhaustive is cheap. If the grid explodes (imagine 4 learning rates × 4 depths × 4 leaf sizes × 4 subsample rates = 256), use `RandomizedSearchCV` with `n_iter=30` or so. Same output format, a fraction of the compute.

---

## 1. Section A — Grid Search as nb08 × a Grid

### 1.1 GridSearchCV on Ridge α against OLS — the California Housing case

This entire section operates on the **California Housing** business case from nb01–nb05: you are still HomeValue Analytics, still predicting median house value, and the CFO is still asking how much regularization is worth carrying. What changes today is the *number of candidates* you can compare in one shot.

**One split, five folds — the pattern from nb08 on.** nb08 ended by splitting each dataset once, 80/20, and doing all model work on `X_train` with five folds. This notebook is the first to work that way from the start: `train_test_split(X, y, test_size=0.20, random_state=RANDOM_SEED)` and nothing else — no validation set, no second split. The 4,128 test tracts are the same rows they have been all term, and they stay locked until nb14's ceremony.

In **nb01's End-to-End Predictive Modeling Workflow**, this section executes **Step 5 (Advanced Models)** + **Step 6 (Model Comparison)** + **Step 7 (Model Selection and Rationale)** at the granularity of the Ridge α dial — running nb08's CV ritual on every candidate and settling the outcome with `compare_paired`.

**Start with what nb08 actually concluded.** In nb08 Exercise 1, Ridge ($\alpha = 1.0$) faced plain OLS on the identical five folds, and `compare_paired` printed *practically equivalent*: the interval around the mean gain was a sliver around zero, far inside the CFO's 0.01 $R^2$ margin. OLS — the model HomeValue runs today, with no hyperparameter to defend — stayed. **That verdict rested on one value of α.** A careful analyst asks: *"Could a different α — 10, 100, 0.01 — show a gain worth having over OLS?"*

**`GridSearchCV`** answers that by sweeping a whole grid of α values, running nb08's 5-fold CV on each, and stacking the results into `cv_results_`. You hand it three things:

1. A **pipeline** (`StandardScaler + Ridge`).
2. A **parameter grid** — `{'ridge__alpha': [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]}`.
3. A **CV splitter** — the same `KFold(5)` object you will also pass to OLS.

Internally it fits 6 α × 5 folds = 30 models, scores each, and stores every fold's score in `split0_test_score` … `split4_test_score`. Because the grid and OLS receive the **same splitter object**, fold $i$ holds the same tracts for every candidate — which is what licenses `compare_paired` later.

**The grid's best cell is a nominee, not a champion.** `best_params_` names the α with the largest mean. That α was picked *because* it scored highest on these five folds, so its mean is a little optimistic, and a larger mean by itself says nothing about whether the gain is worth having. So the plan is:

- Run `GridSearchCV` over the α grid (30 fits) and a 5-fold CV of OLS on the same splitter (5 fits).
- Show all seven candidates' fold scores, means, and SDs in one table and one strip — a description, not a decision.
- Let the grid's best α face OLS with `compare_paired(nominee_folds, ols_scores, DELTA_R2)` — the nominee is A, OLS is B — and run the band. The nominee replaces OLS only on *"A is better by a worthwhile amount"*.

> **An analogy that often helps here.** Imagine nb08's 5-fold CV as a single lab experiment — one recipe, five beakers, one result. `GridSearchCV` is the whole lab running six experiments in parallel (six Ridge recipes), and OLS-with-CV is the recipe the kitchen already uses. All seven results land on the same whiteboard, and because every experiment used the same five beakers, you can compare the best new recipe with the current one beaker by beaker. The new recipe goes on the menu only if it is better by an amount the chef said would matter — not merely because it topped the whiteboard.

> 💡 **Gemini Prompt:** "Load the California Housing dataset and split it once with `train_test_split(X, y, test_size=0.20, random_state=RANDOM_SEED)` into `X_reg_train` and a locked `X_reg_test`. Build `cv_reg = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)` and two pipelines: `StandardScaler + Ridge` and `StandardScaler + LinearRegression` (OLS, HomeValue's incumbent). Run `GridSearchCV` on the Ridge pipeline with `param_grid={'ridge__alpha': [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]}`, `cv=cv_reg`, `scoring='r2'`, and `cross_val_score` on OLS with the same `cv_reg`. Print Ridge's `best_params_` and `best_score_` and OLS's mean and fold scores. Build one table of all seven candidates with their five fold scores (from the `split0..split4_test_score` columns of `cv_results_`, and from `ols_scores`), their mean, and their SD (`ddof=1`) — no interval — sorted by mean. Plot each candidate's five fold scores as small dots and its mean as a large dot (OLS in darkorange). Then treat the best α as a nominee: pull its fold scores at `grid_ridge.best_index_`, call `compare_paired(nominee_folds, ols_scores, DELTA_R2)`, and print the per-fold gains with 6 decimals, the mean gain, the interval, and the outcome; finally call `show_band(nominee_folds, ols_scores, DELTAS_R2, 'Ridge α=<best>', 'OLS')`."
>
> **After running, verify:**
> - The split prints 16,512 training tracts and 4,128 locked test tracts, and there is no validation set
> - The grid fits 6 Ridge × 5 folds = 30 sub-fits, plus one 5-fold OLS run on the same splitter
> - One table prints seven rows with five fold scores, mean, and SD — no interval columns
> - The strip shows seven candidates with five small dots and one large dot each, OLS in darkorange
> - The nominee's per-fold gains, mean gain, interval, and outcome print, followed by one line per margin
> - All numerical outputs use standard decimal format — no scientific notation

In [ ]:
# --- Load California Housing: ONE 80/20 split (the test set stays locked until nb14) ---
cal = fetch_california_housing(as_frame=True)
X_reg = cal.data
y_reg = cal.target

X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=RANDOM_SEED
)
print(f'Train: {len(X_reg_train)} | Test: {len(X_reg_test)} (locked)')

# --- ONE shared splitter for the Ridge grid AND for OLS, so fold i holds the same rows for both ---
cv_reg = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

# --- Ridge α-grid via GridSearchCV ---
ridge_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', Ridge(random_state=RANDOM_SEED))
])
alpha_grid = {'ridge__alpha': [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]}
grid_ridge = GridSearchCV(ridge_pipeline, param_grid=alpha_grid, cv=cv_reg,
                          scoring='r2', return_train_score=False, n_jobs=-1)
grid_ridge.fit(X_reg_train, y_reg_train)
print(f"\nRidge best params: {grid_ridge.best_params_}")
print(f'Ridge best 5-fold CV R^2 (mean): {grid_ridge.best_score_:.4f}')

# --- OLS — HomeValue's incumbent (nb08 kept it) — on the same folds ---
ols_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('ols', LinearRegression())
])
ols_scores = cross_val_score(ols_pipeline, X_reg_train, y_reg_train, cv=cv_reg, scoring='r2', n_jobs=-1)
print(f'\nOLS 5-fold CV R^2 (mean): {ols_scores.mean():.4f}')
print(f'OLS fold scores: {np.round(ols_scores, 4)}')

# --- Every candidate's evidence: five fold scores, their mean and SD (no interval) ---
fold_cols = [f'split{i}_test_score' for i in range(5)]
cvr = pd.DataFrame(grid_ridge.cv_results_).copy()
rows = [{'candidate': f'Ridge α={a}', 'folds': cvr.loc[i, fold_cols].to_numpy(dtype=float)}
        for i, a in enumerate(cvr['param_ridge__alpha'])]
rows.append({'candidate': 'OLS (incumbent)', 'folds': ols_scores})
combined = pd.DataFrame(rows)
combined['mean'] = combined['folds'].apply(np.mean)
combined['sd'] = combined['folds'].apply(lambda f: np.std(f, ddof=1))
combined = combined.sort_values('mean', ascending=False).reset_index(drop=True)
print('\nAll candidates, ranked by mean 5-fold CV R^2 (a description — the ranking decides nothing):\n')
print(combined.assign(folds=combined['folds'].apply(lambda f: ' '.join(f'{x:.4f}' for x in f)))
      [['candidate', 'folds', 'mean', 'sd']].to_string(index=False, float_format=lambda x: f'{x:.4f}'))

# --- Picture: each candidate's five fold scores (small dots) and mean (large dot); OLS in darkorange ---
fig, ax = plt.subplots(figsize=(11, 5))
for i, r in combined.iterrows():
    color = 'darkorange' if r['candidate'].startswith('OLS') else 'steelblue'
    ax.scatter(np.full(5, i), r['folds'], s=25, color=color, alpha=0.6)
    ax.scatter([i], [r['mean']], s=120, color=color, edgecolor='black', zorder=3)
ax.set_xticks(range(len(combined)))
ax.set_xticklabels(combined['candidate'], rotation=20)
ax.set_ylabel('5-fold CV R^2')
ax.set_title('California Housing — Ridge α grid and OLS: five fold scores (small) and their mean (large)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# --- The grid's best cell is a NOMINEE: it faces the incumbent with compare_paired ---
best_alpha = grid_ridge.best_params_['ridge__alpha']
nominee_folds = cvr.loc[grid_ridge.best_index_, fold_cols].to_numpy(dtype=float)
d, gain, (lo, hi), verdict = compare_paired(nominee_folds, ols_scores, DELTA_R2)
print(f'\nNominee Ridge α={best_alpha} (A) vs OLS (B, incumbent) — the same five folds')
print(f'Gain per fold: {[f"{x:+.6f}" for x in d]}')
print(f'Mean gain {gain:+.6f} R^2, interval corrected for fold overlap [{lo:+.6f}, {hi:+.6f}]')
print(f'Outcome at δ = {DELTA_R2}: {verdict}')
show_band(nominee_folds, ols_scores, DELTAS_R2, f'Ridge α={best_alpha}', 'OLS')

**Reading the output:**

Four artifacts to walk through.

**First: `grid_ridge.best_params_` and `best_score_`.** Ridge's best cell by mean is **α = 10.0**, with a mean CV $R^2$ of **0.6045**. Treat it as a *nominee*, not a decision. The double-underscore in `param_ridge__alpha` is sklearn's way of reaching into a named pipeline step (`'ridge'.alpha` becomes `ridge__alpha`).

**Second: the table and the strip.** OLS scores **0.6184, 0.5991, 0.6299, 0.5881, 0.5868** (mean **0.6044**, SD **0.0190**), and every Ridge α from 0.01 to 10 is within 0.0001 of it on the mean; α = 100 slips to 0.6039 and α = 1000 drops to 0.5792, where the penalty finally starts to cost accuracy. The strip shows the same thing visually — the top five candidates are stacked on top of each other, fold by fold. A table like this describes; it does not decide. The ranking at the top is a ranking of differences in the fifth decimal.

**Third: the nominee against OLS.** Scored on the identical five folds, α = 10 minus OLS gives **+0.000092, −0.000039, −0.000162, +0.000172, +0.000027** — a mean gain of **+0.000018** $R^2$ with an interval corrected for fold overlap of **[−0.000220, +0.000255]**. The whole interval sits deep inside $\pm 0.01$, so `compare_paired` prints **"Practically equivalent: choose the simpler model"**, and the band line shows the same outcome at 0.005 and 0.02: **robust across the three declared margins**. OLS stays as HomeValue's champion.

**Fourth: what this adds to nb08.** nb08 compared OLS with one α; today OLS held up against the best of six, and the comparison is a *positive* finding — the evidence rules out a gain the CFO would act on, in either direction. That is exactly the inner cycle of **nb01 Steps 5–7** that you will reuse for every model family from here on: *sweep the dial, show every candidate's fold scores, let the best cell face the current champion with `compare_paired`, and promote it only on a worthwhile gain.*

> **A question that often comes up here:** *"The grid's best α has the highest mean — why isn't it simply the new champion?"* Because a search picks its winner *by* looking at these five folds, so the winner's mean is optimistic by construction: among six nearly identical candidates, one of them had to come out on top, and the one that did got a little lucky on this particular partition. A higher mean also says nothing about whether the gain is large enough to matter to the CFO. So the best cell is a **nominee** — it has to face the model already running, on the same folds, against the margin declared before scoring, and only *"A is better by a worthwhile amount"* gives it the job. On California Housing it does not come close, and OLS keeps the job with nothing to defend.

---

### 1.2 RandomizedSearchCV on MedScreen — when the grid is too big

Section 1.1 ran a tidy `GridSearchCV` on a 6-value $\alpha$ grid for HomeValue's Ridge regression and added OLS as a comparator. That was clean because Ridge has exactly one knob worth tuning. Now turn to **MedScreen**, the Health Department's cancer-screening project from nb06–nb08, and the same machinery faces a heavier grid: classification problems usually expose more knobs than regression, and exhaustive sweeps start to feel expensive.

**Picture sweeping a logistic regression with three knobs you already know.** From nb06 you have `C` (the regularization strength); from nb05's Ridge/Lasso vocabulary you recognize `penalty` (`l1` vs `l2` — the same shrinkage distinction); and the `solver` (the optimization routine sklearn uses to fit the model). A reasonable sweep for MedScreen is **10 values of `C` × 2 penalty types × 2 valid solvers = 40 combinations**, each requiring a 5-fold stratified CV run, so **200 model fits**. On Breast Cancer that runs in well under a minute — the cost is fine here.

The count grows *multiplicatively*, though. Bring in a fourth knob — any extra dimension you might want to sweep — and you multiply the grid by however many values it takes. **40 × 5 = 200 combinations × 5 folds = 1,000 fits**, and that is just for one extra knob; add a fifth and you are at 5,000. On a real-world screening dataset with hundreds of thousands of patient records, that becomes an overnight wait. The danger is not the *first* hyperparameter — it is every one after the first.

`GridSearchCV`'s problem is that it explores every combination *with equal effort*, even ones that are obviously bad. If `C = 1000` produces a near-overfitted classifier (regularization so weak the model chases noise that looks like signal), `GridSearchCV` still runs five CV folds on it before reporting that result. That is wasted compute — and on a medical screening problem where the Health Department's reviewers expect a defensible model, you want to spend that budget where it actually buys evidence.

**`RandomizedSearchCV` solves the budget problem.** Instead of enumerating every grid point, you pass it *distributions* (e.g., `loguniform(1e-3, 1e3)` for the `C` parameter) plus an `n_iter` argument that says "draw this many random samples and try them." Each sample becomes one CV run, exactly like a grid point. The output format is identical — `cv_results_` table, `best_params_`, `best_score_` — so everything you learned in Section 1.1 about reading the table transfers directly. You spend compute *more efficiently*: rather than 200 evenly-spaced fits, you might do 30 well-distributed random fits and land on a winner of similar quality at one-eighth the cost.

The intuition to carry forward: **`GridSearchCV` is a comb, `RandomizedSearchCV` is a fishing net.** A comb is exhaustive but expensive when the space is wide; a net trades exhaustiveness for coverage at low cost. Both pull fish out of the water; you choose based on how big the water is.

#### A quick primer on `C` — what is the parameter you are about to sweep?

Before you run the search, one paragraph on what `C` actually is — because it is named in a way that trips up almost everyone the first time.

**Big idea:** `C` is the *inverse* of regularization strength. Small `C` means *more* regularization; large `C` means *less*. That is the opposite of what your gut probably expects, so it is worth pausing to lock in.

To unpack that: logistic regression learns one coefficient per feature. With no restriction, those coefficients can grow arbitrarily large to fit every twist in the training data — including noise. *Regularization* is the penalty that pulls coefficients back toward zero, trading a sliver of training fit for a lot of stability on data the model has never seen. `C` controls how aggressive that pull is — but inversely.

Three regimes worth keeping in your head:

- **Small `C` (e.g., `C = 0.01`).** Strong regularization → small coefficients → simpler, smoother decision boundary. The model resists chasing individual training points. Less variance from seed to seed, less overfitting risk; sometimes a sliver more bias if real signal gets shrunk too aggressively.
- **Large `C` (e.g., `C = 100`).** Weak regularization → coefficients are free to grow → flexible boundary that hugs the training data closely. That can be exactly what you want, *or* the model memorizing noise.
- **`C = 1.0`.** Sklearn's default. A neutral starting point for most balanced classification problems — when in doubt, this is where you start.

The values you will see throughout this notebook (`0.001, 0.01, 0.1, 1, 10, 100, 1000`) are spaced on a **log scale** on purpose. Each step is 10× more flexible than the previous one — and the *multiplicative* gap is the unit that actually moves the model. Doubling `C` from 1.0 to 2.0 barely changes anything; jumping from 1.0 to 100.0 puts you in a totally different regime. That is exactly why the Gemini Prompt below uses `loguniform(1e-3, 1e3)` rather than a plain `uniform`: log-uniform sampling gives every order of magnitude an equal chance of being explored, which is what you want when the parameter's *effect* is multiplicative.

**Which `C` is "the simpler model"?** In this course the answer is set by the decision, not by the coefficients: the **default `C = 1.0` is MedScreen's incumbent** — the model nb06–nb08 built, explained, and handed to the review board. A tuned `C` is a challenger, and it carries the burden of proof. When `compare_paired` says the two are practically equivalent or the comparison is unresolved, the default stays — *"the gap was too small to matter, so we kept the model the board already reviewed"* — rather than anyone fighting over a 0.0005 mean difference that will not survive a reseed.

**Plan, parallel to Section 1.1.** Section 1.1 let the grid's nominee face OLS. The MedScreen analog lets the random search's nominee face **the default `C = 1.0`** and asks *"did random search find a `C` that is better than the default by a worthwhile amount, or did tuning buy nothing measurable?"* So:

- Split Breast Cancer once, stratified 80/20, and run `RandomizedSearchCV` over `loguniform(1e-3, 1e3)` for `C`, drawing 20 random samples (= 20 × 5 folds = 100 fits) on one `StratifiedKFold` object.
- Run a 5-fold stratified CV of `LogisticRegression(C=1.0, max_iter=5000)` — the default — on the **same** splitter (5 more fits).
- Show all 21 candidates' means and SDs in one table, and the top-5 sampled candidates plus the default as a strip of fold scores (default in `darkorange`).
- Let the search's best `C` face the default with `compare_paired(tuned_folds, default_scores, DELTA_AUC)` and run the band.

The outcome gives the Health Department reviewer either a tuned model with a worthwhile gain to defend it, or a clean justification for keeping the default.

> 💡 **Gemini Prompt:** "On MedScreen's Breast Cancer dataset, split once with `train_test_split(X, y, test_size=0.20, random_state=RANDOM_SEED, stratify=y)` and build `cv_clf = StratifiedKFold(5, shuffle=True, random_state=RANDOM_SEED)`. Build a pipeline `StandardScaler + LogisticRegression(max_iter=5000)` and run `RandomizedSearchCV` with `param_distributions={'clf__C': scipy.stats.loguniform(1e-3, 1e3)}`, `n_iter=20`, `cv=cv_clf`, `scoring='roc_auc'`, `random_state=RANDOM_SEED`; print `best_params_` and `best_score_`. Run `cross_val_score` on `StandardScaler + LogisticRegression(C=1.0, random_state=RANDOM_SEED, max_iter=5000)` — the default, MedScreen's incumbent — with the same `cv_clf`, and print its mean and fold scores. Build one table of all 21 candidates (20 sampled + the default, labelled `'LogReg C=1.0 (default, incumbent)'`) with each one's five fold scores, mean, and SD (`ddof=1`) — no interval — sorted by mean. Plot the top-5 sampled candidates plus the default as a strip: five small dots and one large mean dot each, the default in darkorange. Then treat the best `C` as a nominee: pull its fold scores at `rand_search.best_index_`, call `compare_paired(tuned_folds, default_scores, DELTA_AUC)`, and print the per-fold gains, mean gain, interval, and outcome; finally call `show_band(tuned_folds, default_scores, DELTAS_AUC, 'C=<best>', 'C=1.0')`."
>
> **After running, verify:**
> - The split prints 455 training patients and 114 locked test patients
> - 20 random `C` values were sampled on a log scale, and the default was evaluated on the same `cv_clf` splitter
> - The table has 21 rows with mean and SD — no interval columns
> - The strip shows six candidates, the default in darkorange
> - The nominee's per-fold gains, mean gain, interval, and outcome print, followed by one line per margin
> - All numerical outputs use standard decimal format — no scientific notation

In [ ]:
# --- Breast Cancer: ONE stratified 80/20 split ---
bc = load_breast_cancer(as_frame=True)
X_clf = bc.data
y_clf = bc.target

X_clf_train, X_clf_test, y_clf_train, y_clf_test = train_test_split(
    X_clf, y_clf, test_size=0.20, random_state=RANDOM_SEED, stratify=y_clf
)
print(f'Train: {len(X_clf_train)} | Test: {len(X_clf_test)} (locked)')

# --- ONE shared splitter for the random search and for the default C=1.0 ---
cv_clf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

# --- Randomized search over C ---
log_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(random_state=RANDOM_SEED, max_iter=5000))
])
rand_search = RandomizedSearchCV(log_pipeline, param_distributions={'clf__C': stats.loguniform(1e-3, 1e3)},
                                 n_iter=20, cv=cv_clf, scoring='roc_auc', random_state=RANDOM_SEED, n_jobs=-1)
rand_search.fit(X_clf_train, y_clf_train)
print(f'\nBest params: {rand_search.best_params_}')
print(f'Best 5-fold CV ROC-AUC (mean): {rand_search.best_score_:.4f}')

# --- The default C=1.0 — MedScreen's incumbent from nb06-nb08 — on the same folds ---
default_logreg_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(C=1.0, random_state=RANDOM_SEED, max_iter=5000))
])
default_scores = cross_val_score(default_logreg_pipeline, X_clf_train, y_clf_train,
                                 cv=cv_clf, scoring='roc_auc', n_jobs=-1)
print(f'\nDefault LogReg (C=1.0) 5-fold CV ROC-AUC (mean): {default_scores.mean():.4f}')
print(f'Default fold scores: {np.round(default_scores, 4)}')

# --- Fold scores, mean, SD for all 21 candidates (20 sampled + the default) ---
fold_cols_clf = [f'split{i}_test_score' for i in range(5)]
cvr_clf = pd.DataFrame(rand_search.cv_results_).copy()
rows_clf = [{'candidate': f'LogReg C={c:.3g}', 'folds': cvr_clf.loc[i, fold_cols_clf].to_numpy(dtype=float)}
            for i, c in enumerate(cvr_clf['param_clf__C'])]
rows_clf.append({'candidate': 'LogReg C=1.0 (default, incumbent)', 'folds': default_scores})
combined_clf = pd.DataFrame(rows_clf)
combined_clf['mean'] = combined_clf['folds'].apply(np.mean)
combined_clf['sd'] = combined_clf['folds'].apply(lambda f: np.std(f, ddof=1))
combined_clf = combined_clf.sort_values('mean', ascending=False).reset_index(drop=True)
print('\nAll 21 candidates ranked by mean 5-fold CV ROC-AUC (a description):')
print(combined_clf[['candidate', 'mean', 'sd']].to_string(index=False, float_format=lambda x: f'{x:.4f}'))

# --- Picture: the top-5 sampled candidates plus the default, fold scores and mean ---
plot_df = pd.concat([combined_clf[~combined_clf['candidate'].str.contains('default')].head(5),
                     combined_clf[combined_clf['candidate'].str.contains('default')]]
                    ).sort_values('mean', ascending=False).reset_index(drop=True)
fig, ax = plt.subplots(figsize=(11, 5))
for i, r in plot_df.iterrows():
    color = 'darkorange' if 'default' in r['candidate'] else 'steelblue'
    ax.scatter(np.full(5, i), r['folds'], s=25, color=color, alpha=0.6)
    ax.scatter([i], [r['mean']], s=120, color=color, edgecolor='black', zorder=3)
ax.set_xticks(range(len(plot_df)))
ax.set_xticklabels(plot_df['candidate'], rotation=20)
ax.set_ylabel('5-fold CV ROC-AUC')
ax.set_title('MedScreen — top-5 sampled C values and the default: fold scores (small) and mean (large)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# --- The search's best C is a NOMINEE: it faces the default with compare_paired ---
best_C = rand_search.best_params_['clf__C']
tuned_folds = cvr_clf.loc[rand_search.best_index_, fold_cols_clf].to_numpy(dtype=float)
d_clf, gain_clf, (lo_clf, hi_clf), verdict_clf = compare_paired(tuned_folds, default_scores, DELTA_AUC)
print(f'\nNominee C={best_C:.3g} (A) vs the default C=1.0 (B, incumbent) — the same five folds')
print(f'Gain per fold: {[f"{x:+.4f}" for x in d_clf]}')
print(f'Mean gain {gain_clf:+.4f} ROC-AUC, interval corrected for fold overlap [{lo_clf:+.4f}, {hi_clf:+.4f}]')
print(f'Outcome at δ = {DELTA_AUC}: {verdict_clf}')
show_band(tuned_folds, default_scores, DELTAS_AUC, f'C={best_C:.3g}', 'C=1.0')

**Reading the output:**

Four artifacts, mirroring §1.1 with the default `C = 1.0` in the role OLS played for HomeValue.

**First: `rand_search.best_params_` and `best_score_`.** RandomizedSearchCV drew 20 `C` values from `loguniform(1e-3, 1e3)` and ran the full 5-fold stratified CV on each. The best by mean is **$C \approx 0.0789$** (mean ROC-AUC **0.9945**) — not a round number, which a fixed grid of round values would never have tried. That is the pitch for randomized search: you explore the space more cleverly than a regular grid at the same compute budget.

**Second: the default and the table.** The default `C = 1.0` scores **0.9923, 0.9969, 0.9995, 0.9995, 0.9825** (mean **0.9941**, SD **0.0072**). Scan the table: the six best sampled values, from $C \approx 0.08$ to $C \approx 0.25$, sit within about 0.0002 of one another, and the default sits right behind them; very large `C` values fall to about 0.985. On a dataset this clean, logistic regression saturates the metric across a wide range of regularization.

**Third: the strip.** Five small dots per candidate, nearly on top of each other fold by fold: a hard fold (the one near 0.98) is hard for every `C`. That shared difficulty is why the comparison is paired.

**Fourth: the nominee against the default.** $C \approx 0.0789$ minus the default, fold by fold: **+0.0005, +0.0031, +0.0005, +0.0000, −0.0021** — mean gain **+0.0004** ROC-AUC, interval corrected for fold overlap **[−0.0030, +0.0038]**, entirely inside $\pm 0.01$: **practically equivalent**, at all three margins. **The default `C = 1.0` stays.** The tuned value bought nothing the review board would act on, and "we kept the model you already reviewed" is the simplest sentence to defend.

The procedure — *random search over `C`, the default on the same folds, every candidate's spread shown, then the nominee against the default with `compare_paired`* — is the inner cycle of **nb01 Steps 5–7** applied to classification, the same way §1.1 applied it to regression. When you quote the champion's score, quote it the nb08 way: *"5-fold CV ROC-AUC 0.9941 (fold SD 0.0072)"*, never as a single number with no spread attached.

> **A question that often comes up here:** *"Is `RandomizedSearchCV` always better than `GridSearchCV`?"* No — for small grids (under \~50 combinations) `GridSearchCV` is both cheaper and more thorough, because it tries everything. `RandomizedSearchCV` wins when the grid gets large enough that exhaustive enumeration stops being free. A good heuristic: start with `GridSearchCV` while the grid has only one or two dimensions, and switch to `RandomizedSearchCV` the moment you add a third dimension — the total count grows multiplicatively. Either way, its best row is a nominee.

---

## 📝 PAUSE-AND-DO Exercise 1 (10 minutes)

**Task:** Run `GridSearchCV` on MedScreen's Logistic Regression over an explicit `C` grid, and decide with `compare_paired` whether the grid's best `C` replaces the default `C = 1.0`.

This exercise applies the pattern of Sections 1.1–1.2: a search, a table of fold scores, a nominee, and a paired comparison with the current setting. Build the same `StandardScaler + LogisticRegression` pipeline as Section 1.2, but use a clean `GridSearchCV` so you can see every candidate. Fit on `X_clf_train` / `y_clf_train` with the same `cv_clf` splitter and `scoring='roc_auc'`.

Use this explicit grid:

```python
param_grid = {'clf__C': [0.01, 0.1, 1.0, 10.0, 100.0]}
```

After fitting, build a table of every row's five fold scores (`split0..split4_test_score`), mean, and SD, sorted by mean. The **nominee** is the top row. If the nominee is `C = 1.0` itself, there is nothing to compare and the default stays. Otherwise call `compare_paired(nominee_folds, default_folds, DELTA_AUC)` — the nominee is A, the default is B — and run `show_band`. The nominee replaces `C = 1.0` only on *"A is better by a worthwhile amount"*. Print the champion.

> **Reminder on the decision rule from nb08 and Section 1.2:** the grid's top row was picked *because* it scored highest on these folds, so it starts with an optimistic edge, and a higher mean by 0.0003 is not a reason to change anything. Practically equivalent, unresolved, or worse: the default stays, because it is the model the review board already knows.

> 💡 **Gemini Prompt:** "Build a Logistic Regression pipeline (`StandardScaler` + `LogisticRegression(random_state=RANDOM_SEED, max_iter=5000)`). Using `X_clf_train`, `y_clf_train`, and `cv_clf` with `scoring='roc_auc'`, run `GridSearchCV` over `param_grid = {'clf__C': [0.01, 0.1, 1.0, 10.0, 100.0]}`. Convert `cv_results_` to a DataFrame; for every row keep the five fold scores from `split0_test_score`..`split4_test_score`, their mean, and their SD (`ddof=1`); sort by mean and print the table. Plot each `C`'s five fold scores as small dots and its mean as a large dot, with `C=1.0` in darkorange. Take the top row as the nominee. If it is `C=1.0`, print that the default stays. Otherwise call `compare_paired(nominee_folds, default_folds, DELTA_AUC)` with the `C=1.0` row's fold scores as B, print the per-fold gains, mean gain, interval, and outcome, and call `show_band`. Set `champion` to the nominee's `C` only if the outcome starts with 'A is better', otherwise to 1.0, and print it."
>
> **After running, verify:**
> - A table prints with each `C`'s five fold scores, mean, and SD — no interval columns
> - The strip has five candidates, `C=1.0` in darkorange
> - Either a 'the default stays' line prints, or the nominee's per-fold gains, interval, and outcome print with one line per margin
> - A final line names the champion `C`
> - All numerical outputs use standard decimal format — no scientific notation

---

In [ ]:
# YOUR SOLUTION CODE HERE
# Hint: Use the Gemini prompt above for step-by-step guidance.
#
# What to do:
#   1. Build pipeline: StandardScaler + LogisticRegression(random_state=RANDOM_SEED, max_iter=5000)
#   2. Run GridSearchCV with param_grid={'clf__C': [0.01, 0.1, 1.0, 10.0, 100.0]}
#      on X_clf_train, y_clf_train with cv=cv_clf and scoring='roc_auc'
#   3. From cv_results_, keep each row's five fold scores (split0..split4), mean, and SD; sort by mean
#   4. Nominee = the top row. If it is C=1.0, the default stays.
#   5. Otherwise: compare_paired(nominee_folds, default_folds, DELTA_AUC) and show_band(...)
#   6. champion = the nominee's C only if the outcome starts with 'A is better'; otherwise 1.0

### YOUR ANALYSIS:

**Question 1:** Which `C` has the highest mean 5-fold CV ROC-AUC, and why is that row only a nominee?
[Your answer]

**Question 2:** Which of the four outcomes does `compare_paired` print for the nominee against `C = 1.0`, and does it change at the small or large margin?
[Your answer]

**Question 3:** Which `C` do you recommend to the Health Department, and why? Use the interval and the margin in your justification.
[Your answer]

---

## 2. Section B — Feature Engineering, Categorical Data, and the Leakage Trap

Section A turned the CV ritual into a ranked table of candidates — one tool, one decision rule. Section B takes on a completely different problem: getting raw, messy, real-world data into a pipeline *safely*.

### 2.1 TechCorp Talent Analytics — the business case

**TechCorp** is a fictional 1,500-person SaaS company you are being loaned to for a single consulting project. Their People Analytics team has a standing ask from the CHRO (Chief Human Resources Officer): *flag employees at high risk of resigning within the next 6 months so HR can run targeted retention conversations before the employee is already out the door*. The math is crisp: losing a mid-level engineer costs the company roughly USD 75,000 (recruitment fees + 6 months of productivity loss + ramp time for the replacement); a retention conversation costs about USD 500 (manager time + modest retention bonus). Even a modest lift over *"managers guessing"* pays for itself many times over.

The People Analytics team built a dataset with the following columns:

- **Numeric features:** `years_at_company`, `satisfaction_score` (a 1–10 pulse-survey result), `salary_pct_of_market` (the company's salary as a percentage of the market median — above 100 is generous, below is underpaying), `projects_completed_last_year`, `manager_interactions_last_quarter`.
- **Low-cardinality categorical features:** `department` (5 levels — Engineering, Sales, Marketing, Operations, Support), `job_level` (5 levels — IC1, IC2, IC3, Manager, Director), `remote_status` (3 levels — onsite, hybrid, remote).
- **High-cardinality categorical:** `manager_id` (80 distinct managers, roughly 25 employees per manager). This one is going to matter a lot in Section 2.3.
- **Target:** `left_within_6mo` (binary — 1 if the employee resigned within 6 months of the snapshot, else 0).

**Why a made-up company instead of real data?** Because the machinery of Section 2 needs a dataset whose structure you control. The dramatic target-encoding leak in Section 2.3 only bites clearly when the dataset has a high-cardinality categorical with small group sizes (that is what `manager_id` with 80 levels and \~25 employees per level is engineered to produce). On a noisy real dataset the leak is often there but the signal/noise ratio can hide it. Here you will see it unambiguously.

### The CHRO's margins — declared now, before any TechCorp model is scored

Every comparison in Section B uses ROC-AUC and the CHRO's three margins (`DELTAS_AUC` in the setup cell): **base 0.01 ROC-AUC** decides; **small 0.005** is for a CHRO who adopts any feature that measurably ranks leavers better; **large 0.02** is for a CHRO who wants a clear lift before HR starts collecting and maintaining a new field. One caution: ROC-AUC summarizes the ranking over every possible cut-off, so it does not tell you exactly who lands on a call list of fixed size — if HR will call the top 50 names each month, check the top of the list separately.

### What makes TechCorp special in this course

TechCorp is the **first dataset you will encounter with real categorical columns**. Both California Housing (nb01–nb05) and Breast Cancer (nb06–nb08) are entirely numeric — every feature is a floating-point number. `ColumnTransformer` has been on the syllabus since nb02 but you have never actually exercised it on live categorical data because there was no categorical data to exercise it on. That changes today. Feeding a column of strings like `['Engineering', 'Sales', 'Marketing']` straight into `LogisticRegression` would raise `could not convert string to float` — you have to *encode* it first, and the professional way to encode is `OneHotEncoder(handle_unknown='ignore')` living inside a `ColumnTransformer` living inside a `Pipeline`. That three-level nesting is today's new pattern.

> **A question that often comes up here:** *"If I can encode categoricals with pandas `get_dummies`, why do I need `ColumnTransformer` and `OneHotEncoder` at all?"* You can — for exploration. For a deployable pipeline, you cannot. `pd.get_dummies` builds the columns once on the DataFrame in front of it, and if a new category appears in production data (a new department, a new manager hired after you trained), the encoded DataFrame has the wrong columns and the model silently breaks. `OneHotEncoder(handle_unknown='ignore')` inside a `Pipeline` learns the vocabulary from training data, transforms held-out folds, test, and production data consistently, and emits all-zero indicators for unseen categories instead of crashing. The discipline of "encoders live inside the Pipeline" is the same discipline of "scalers live inside the Pipeline" — if the transform looks at the data, it belongs inside the object the CV wraps.

> 💡 **Gemini Prompt:** "Generate a synthetic TechCorp attrition dataset with 2000 rows. Use `numpy.random` seeded at `RANDOM_SEED`. Build a pandas DataFrame with: numeric columns `years_at_company` (integer 0–15), `satisfaction_score` (1–10), `salary_pct_of_market` (60–140), `projects_completed_last_year` (0–12), `manager_interactions_last_quarter` (0–20); categorical columns `department` (one of Engineering/Sales/Marketing/Operations/Support), `job_level` (IC1/IC2/IC3/Manager/Director), `remote_status` (onsite/hybrid/remote), `manager_id` (80 distinct managers — one per \~25 employees). Create a binary target `left_within_6mo` whose probability is higher when `satisfaction_score` is low, `salary_pct_of_market` is low, `manager_interactions_last_quarter` is low, `remote_status == 'onsite'`, or `department == 'Support'`. Target around 22% positive rate. Also add 30 noise columns named `engagement_metric_00` through `engagement_metric_29` drawn from `rng.normal(0, 1)` — TechCorp's HRIS really does export this many engagement fields, and they matter for Section 2.3's leakage demo. Print the head, the class balance, and the dtypes so the numeric vs categorical split is obvious."

In [ ]:
# --- Generate synthetic TechCorp attrition dataset ---
rng = np.random.default_rng(RANDOM_SEED)
n = 2000

years = rng.integers(0, 16, size=n)
satisfaction = rng.integers(1, 11, size=n)
salary_pct = rng.uniform(60, 140, size=n).round(1)
projects = rng.integers(0, 13, size=n)
mgr_interactions = rng.integers(0, 21, size=n)

department = rng.choice(
    ['Engineering', 'Sales', 'Marketing', 'Operations', 'Support'],
    size=n, p=[0.35, 0.20, 0.15, 0.15, 0.15]
)
job_level = rng.choice(
    ['IC1', 'IC2', 'IC3', 'Manager', 'Director'],
    size=n, p=[0.25, 0.30, 0.25, 0.15, 0.05]
)
remote_status = rng.choice(
    ['onsite', 'hybrid', 'remote'],
    size=n, p=[0.30, 0.45, 0.25]
)

# --- High-cardinality categorical: 80 manager IDs (one manager per ~25 employees) ---
manager_id = rng.choice([f'MGR_{i:03d}' for i in range(80)], size=n)

# --- Attrition propensity ---
logit = (
    -2.0
    + 0.35 * (10 - satisfaction)
    + 0.025 * (100 - salary_pct)
    + 0.08 * (10 - mgr_interactions)
    + 0.40 * (remote_status == 'onsite').astype(float)
    + 0.35 * (department == 'Support').astype(float)
    - 0.05 * years
)
prob_leave = 1.0 / (1.0 + np.exp(-logit))
left_within_6mo = (rng.uniform(0, 1, size=n) < prob_leave).astype(int)

techcorp = pd.DataFrame({
    'years_at_company': years,
    'satisfaction_score': satisfaction,
    'salary_pct_of_market': salary_pct,
    'projects_completed_last_year': projects,
    'manager_interactions_last_quarter': mgr_interactions,
    'department': department,
    'job_level': job_level,
    'remote_status': remote_status,
    'manager_id': manager_id,
    'left_within_6mo': left_within_6mo
})

# --- 30 noise columns (TechCorp's HRIS exports many engagement telemetry metrics; most are noise) ---
# These columns are included because TechCorp's HR system really does export dozens of fields.
# They also make the selection-bias leak in Section 2.3 clearly visible.
NOISE_COLS = [f'engagement_metric_{j:02d}' for j in range(30)]
for col in NOISE_COLS:
    techcorp[col] = rng.normal(0.0, 1.0, size=n)

# --- Inspect the dataset: head, class balance, dtypes (all 40 columns) ---
print(f'TechCorp shape: {techcorp.shape}  (10 core columns + {len(NOISE_COLS)} engagement-metric noise cols)')
print()
print('Class balance (left_within_6mo):')
print(techcorp['left_within_6mo'].value_counts().rename({0: 'stayed (0)', 1: 'left (1)'}).to_string())
print(f"  positive rate: {techcorp['left_within_6mo'].mean():.1%}")
print()
print('Dtypes (all columns — numeric vs object/categorical split is obvious):')
print(techcorp.dtypes.to_string())
print()
print('TechCorp head (all columns):')
techcorp.head()

**Reading the output:**

The dataframe prints 2,000 rows. Three of the columns (`department`, `job_level`, `remote_status`) show up as `object` dtype — pandas' way of saying *"text labels, not numbers."* That is the moment this course has been working toward: every nb01–nb08 dataset was entirely numeric, so you could get away with `StandardScaler` and nothing else. TechCorp breaks that pattern. Feeding an `object`-typed column straight into `LogisticRegression` would raise a "could not convert string to float" error — which is the problem `ColumnTransformer` solves by routing numeric and categorical columns to different preprocessing branches.

The attrition rate prints at **37.7%** (the synthetic generator is mildly aggressive on purpose, so the positive class has enough mass for stable CV). That is more imbalanced than Breast Cancer (63/37) but less imbalanced than a typical fraud or churn dataset (95/5). You now know exactly how to handle it — keep `stratify=y` in every split, and watch both ROC-AUC (threshold-free, our default) and the confusion matrix at your chosen threshold.

The 30 `engagement_metric_*` columns are pure noise, drawn from a standard normal. They are here because real HR data really does export dozens of low-signal telemetry fields — and they make the selection-bias leak in Exercise 2 clearly visible. The signal-to-noise ratio in the feature space is exactly what makes `SelectKBest`-outside-the-pipeline a dangerous default.

> **A question that often comes up here:** *"The attrition rate is 37.7%, not the 22% I was expecting from the Gemini prompt."* The generator's logit weights push a bit higher than the target rate in this seed — which is still fine for our purposes because the dataset stays solidly balanced enough for stratified CV to work cleanly. Class balance matters less than class *presence in every fold*, which is what `StratifiedKFold` guarantees.

---

### 2.2 ColumnTransformer — mixing numeric and categorical preprocessing

You have a dataset with two kinds of columns — numeric (five core + 30 noise) and categorical (four columns, one of them high-cardinality). Numeric columns want `StandardScaler` (they need centering and scaling so LogReg's coefficients have comparable magnitudes). Categorical columns want `OneHotEncoder` (they cannot be fed to a linear model as strings). You need **both**, and you need both to be applied *inside the same pipeline* so that every CV fold refits them on its own training portion only.

`ColumnTransformer` is the sklearn object that lets you do this. Think of it as two parallel assembly lines that start with the same raw DataFrame and end by concatenating their outputs into one wide feature matrix:

```
                    [X raw DataFrame]
                          |
              +-----------+-----------+
              |                       |
         numeric branch          categorical branch
         StandardScaler          OneHotEncoder
         on NUM_COLS             on CAT_COLS
              |                       |
              +-----------+-----------+
                          |
                 [X preprocessed, concatenated]
                          |
                  LogisticRegression
```

In code, this is one constructor call:

```python
preprocess = ColumnTransformer([
    ('num', StandardScaler(), NUM_COLS),
    ('cat', OneHotEncoder(handle_unknown='ignore'), CAT_COLS)
])
```

Each tuple is `(name, transformer, list_of_columns)`. The `name` is just a label sklearn uses in `cv_results_` (e.g., `num__with_mean` if you later tune it). The transformer is the step that branch applies. The list of columns tells `ColumnTransformer` which columns of the incoming DataFrame flow into that branch. `NUM_COLS` and `CAT_COLS` are disjoint — no column goes through both branches — so the outputs concatenate cleanly.

Then you wrap the whole thing in a `Pipeline`:

```python
tc_pipeline = Pipeline([
    ('prep', preprocess),
    ('clf', LogisticRegression(random_state=RANDOM_SEED, max_iter=5000))
])
```

Now `cross_val_score(tc_pipeline, X_tc_train, y_tc_train, cv=cv_tc, scoring='roc_auc')` does the right thing on every fold: the scaler's means are learned from each fold's training portion only, the one-hot vocabulary is learned from each fold's training portion only, and none of the held-out-fold data contaminates the fit. That is what "leak-free" means in concrete terms.

**The `handle_unknown='ignore'` flag is non-negotiable.** If a prediction-time row shows a category that was never in the training portion of the fold — a new hire in a brand-new department, a manager hired after your data snapshot — the encoder has two choices: raise an exception (default) or emit a row of zeros for that category block (with `handle_unknown='ignore'`). The second option is the only one compatible with production deployment, where you cannot predict the full set of categories in advance.

> 💡 **Gemini Prompt:** "Split TechCorp into X (features) and y (target `left_within_6mo`). Split once with `train_test_split(X, y, test_size=0.20, random_state=RANDOM_SEED, stratify=y)` into `X_tc_train` and a locked `X_tc_test`. Build a `ColumnTransformer` with two branches: (a) `StandardScaler` on the 5 core numeric columns plus the 30 engagement noise columns, (b) `OneHotEncoder(handle_unknown='ignore')` on the 4 categorical columns. Wrap it in a `Pipeline` together with `LogisticRegression(random_state=RANDOM_SEED, max_iter=5000)`. Run 5-fold stratified `cross_val_score` on `X_train`, `y_train` with `scoring='roc_auc'`. Use `cv_tc = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)` and keep that object — every TechCorp comparison today reuses it. Print the fold scores, mean, and SD (`ddof=1`) — no interval. Label the result `TechCorp baseline — leak-free pipeline`."

In [ ]:
# --- ONE stratified 80/20 split ---
CORE_NUM_COLS = ['years_at_company', 'satisfaction_score', 'salary_pct_of_market',
                 'projects_completed_last_year', 'manager_interactions_last_quarter']
NUM_COLS = CORE_NUM_COLS + NOISE_COLS  # feed all numeric columns into the pipeline
CAT_COLS_LOW_CARD = ['department', 'job_level', 'remote_status']
CAT_COLS = CAT_COLS_LOW_CARD + ['manager_id']  # manager_id is high-cardinality (80 levels)

X_tc = techcorp[NUM_COLS + CAT_COLS]
y_tc = techcorp['left_within_6mo']

X_tc_train, X_tc_test, y_tc_train, y_tc_test = train_test_split(
    X_tc, y_tc, test_size=0.20, random_state=RANDOM_SEED, stratify=y_tc
)
print(f'Train: {len(X_tc_train)} | Test: {len(X_tc_test)} (locked)')

# --- ColumnTransformer + LogReg pipeline ---
preprocess = ColumnTransformer([
    ('num', StandardScaler(), NUM_COLS),
    ('cat', OneHotEncoder(handle_unknown='ignore'), CAT_COLS)
])
tc_pipeline = Pipeline([
    ('prep', preprocess),
    ('clf', LogisticRegression(random_state=RANDOM_SEED, max_iter=5000))
])

# --- 5-fold stratified CV: fold scores, mean, SD ---
cv_tc = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
cv_scores_tc = cross_val_score(tc_pipeline, X_tc_train, y_tc_train, cv=cv_tc, scoring='roc_auc')
mean_tc = cv_scores_tc.mean()
sd_tc = cv_scores_tc.std(ddof=1)

print('\nTechCorp baseline — leak-free pipeline')
print(f'Fold AUC: {np.round(cv_scores_tc, 4)}')
print(f'Mean AUC: {mean_tc:.4f}')
print(f'SD (k-1): {sd_tc:.4f}')

**Reading the output:**

The TechCorp baseline pipeline prints five fold scores (**0.7870, 0.7640, 0.7566, 0.7885, 0.7351**), their mean (**0.7663**), and their SD (**0.0223**) on the 1,600 training employees — not as sharp as MedScreen's 0.99, because HR data is genuinely noisier than medical imaging, but solidly better than the "managers guessing" baseline (ROC-AUC = 0.50 by definition, the coin-flip ceiling).

The crucial property of this pipeline is that **every statistic — standardization means, one-hot vocabularies — is computed inside each CV fold's training portion only**. Nothing the pipeline learns about column means or category levels is influenced by the held-out fold. That is what makes this number trustworthy. This leak-free pipeline is also TechCorp's **current champion**: every change in Section B — a different encoding, a new feature — is a challenger that has to beat it with `compare_paired` on these same five folds. Sections 2.3 and Exercise 2 are about to violate the leak-free property on purpose, so you can see the inflated score that results.

Two practical notes on what just happened under the hood of the one `ColumnTransformer` call:

- The **numeric branch** applied `StandardScaler` to 35 columns (5 core + 30 noise engagement metrics). Every fold sees its own mean and SD, not the full dataset's.
- The **categorical branch** applied `OneHotEncoder(handle_unknown='ignore')` to 4 columns — including `manager_id` with its 80 levels. That alone adds 80 sparse indicator columns. LogReg with default L2 handles this cheerfully. If you had 8,000 managers instead of 80, you would want either a low-cardinality re-bucketing or a properly-wrapped target encoder — not raw OHE.

> **A question that often comes up here:** *"Does the order of preprocessing inside `ColumnTransformer` matter?"* For `StandardScaler` on numerics and `OneHotEncoder` on categoricals, no — the two branches run in parallel on disjoint columns and the outputs are concatenated. Order matters *within* a branch (e.g., impute before scale, not after), and order matters in the outer `Pipeline` (preprocess before model). Inside `ColumnTransformer`'s branches, the order of the list of `(name, transformer, cols)` tuples is cosmetic only.

---

### 2.3 The intern's leaky pipeline — target encoding on full data

Now the dramatic moment of the notebook. While you were out of the office, an intern on the People Analytics team shipped a "first cut" of the TechCorp model. Their CV ROC-AUC came out *notably* higher than the baseline you just computed. They were celebrating. The CHRO wants to see the numbers at the Monday meeting.

The intern reached for a Kaggle trick they had read about: **target encoding**. Before you dive into what went wrong, let us spend a minute on what target encoding *is*, because many undergrads meet it for the first time today.

### What target encoding is (and why it is tempting)

Suppose you have a categorical column like `manager_id` with 80 levels. The obvious encoding is one-hot: create 80 indicator columns, one per manager, each a 0/1. That works, but it has two practical downsides. It produces a **wide, sparse feature matrix** (80 mostly-zero columns), and it treats every manager as equally unknown — the model has to learn a separate coefficient for each manager from scratch, with only \~25 employees per manager to learn from. That is a lot of parameters for a little bit of signal.

**Target encoding** is the shortcut. Instead of one-hot encoding the categorical column into 80 indicators, replace each category with the **mean of the target within that category**. So for `manager_id = MGR_017`, you compute the fraction of employees under MGR_017 who resigned, and that fraction becomes the encoded value. 80 levels collapse into **one dense numeric column** that already contains the signal "how attrition-prone are this manager's employees?". In practice, on Kaggle, target encoding can give a real boost on high-cardinality categoricals — *when it is done right*.

The intern applied it to all four categorical columns: `department`, `job_level`, `remote_status`, and `manager_id`. Here is their code:

```python
# Intern's approach
techcorp_te = techcorp.copy()
for col in CAT_COLS:  # includes manager_id
    techcorp_te[col + '_te'] = (
        techcorp_te.groupby(col)['left_within_6mo'].transform('mean')
    )
    techcorp_te = techcorp_te.drop(columns=[col])
```

### Why this leaks (the mechanical reason)

Read the first line of the loop very carefully: `techcorp_te.groupby(col)['left_within_6mo'].transform('mean')`. That groups the **full dataset** by the categorical column and computes each group's mean of the **target**, then assigns the group-mean back to every row in that group.

*Every row's label is used in the computation.* Training rows, the rows each CV fold will hold out, test rows — all of them contribute to each group's mean. So the encoded column `manager_id_te` for a specific employee contains the average resignation rate for that employee's manager, computed using — among others — *that employee's own label*.

Then the intern does a train/test split and runs 5-fold CV on the encoded matrix. Every CV fold's held-out rows were already used to compute the encoded values in the training portion. The classifier is training on a feature that carries information about the held-out rows' labels.

### Why it bites so hard on `manager_id`

This is the part that often feels most counterintuitive. On a low-cardinality categorical like `department` (5 levels, \~400 rows per level), the group means are averages of many rows. One training row's label barely moves the group mean. The leak is still there but it is small.

On `manager_id` with 80 levels and \~25 employees per level, **each group mean is an average of only \~25 labels**. A single training row's label shifts the group mean by roughly 1/25 = 4%. When CV later holds out 5 of those 25 rows as the held-out fold, the encoded value for those 5 rows is still carrying information about their own labels — the training rows' encoded values were computed *with those labels in the mean*. The encoded column is nearly a direct readout of the target for the rows you are about to predict.

The smaller the group size, the more severe the leak. `manager_id` with 25-row groups is small enough that the leak is dramatic and easy to see.

### The cell below

Run the intern's code as they wrote it, then compare the result with the leak-free Section 2.2 baseline on the **same five folds** — the same split call and the same `cv_tc`, so fold $i$ holds the same employees in both runs. Expect the leaky version to come in several hundredths higher. That gap is the *size of the lie*: the number the intern's cell prints is not a generalization estimate, it is a partial memorization of the training labels. Notice what the course's own comparison tool says about it, too — the leak is big enough that `compare_paired` would happily call it a worthwhile gain, which is exactly why leaks are dangerous: the decision rule can only judge the numbers it is given.

> 💡 **Gemini Prompt:** "Reproduce the intern's target-encoded pipeline. Copy `techcorp` to `techcorp_te`, then for each categorical column in `CAT_COLS` (which includes the high-cardinality `manager_id`), create `<col>_te = techcorp_te.groupby(col)['left_within_6mo'].transform('mean')` and drop the original `<col>`. Split `techcorp_te` into `X_te` and `y_te`, and apply the same single split as Section 2.2 — `train_test_split(X_te, y_te, test_size=0.20, random_state=RANDOM_SEED, stratify=y_te)` — keeping `Xte_train`, `yte_train`; assert that `Xte_train.index` equals `X_tc_train.index`. Build `leaky_pipeline = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(random_state=RANDOM_SEED, max_iter=5000))])` and run `cross_val_score` on `Xte_train`, `yte_train` with `cv=cv_tc` and `scoring='roc_auc'`. Print both runs' fold scores, means, and SDs. Then call `compare_paired(cv_leaky, cv_scores_tc, DELTA_AUC)` and print the per-fold inflation, its mean, its interval, and the outcome. Plot the two runs' fold scores side by side with a line joining each fold's pair (leak-free in steelblue, leaky in crimson), and end with the fix: drop the target-encoded columns and fall back to `OneHotEncoder(handle_unknown='ignore')` inside the `Pipeline`."
>
> **After running, verify:**
> - The assert passes: both runs use the same training rows and the same folds
> - The leaky mean prints a few hundredths above the leak-free baseline, and the per-fold inflation is positive
> - `compare_paired` prints the inflation's interval and outcome
> - Most of the leak comes from `manager_id_te` (80 levels with \~25 employees each)
> - All numerical outputs use standard decimal format — no scientific notation

In [ ]:
# --- Intern's leaky pipeline: target encoding on the FULL dataset ---
techcorp_te = techcorp.copy()
for col in CAT_COLS:  # includes manager_id (80 levels)
    techcorp_te[col + '_te'] = (
        techcorp_te.groupby(col)['left_within_6mo'].transform('mean')
    )
    techcorp_te = techcorp_te.drop(columns=[col])

X_te = techcorp_te.drop(columns=['left_within_6mo'])
y_te = techcorp_te['left_within_6mo']

# Same single 80/20 split call as Section 2.2, so the training rows — and the five folds — are the same rows
Xte_train, Xte_test, yte_train, yte_test = train_test_split(
    X_te, y_te, test_size=0.20, random_state=RANDOM_SEED, stratify=y_te
)
assert Xte_train.index.equals(X_tc_train.index), 'the two runs must use the same training rows'

leaky_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(random_state=RANDOM_SEED, max_iter=5000))
])
cv_leaky = cross_val_score(leaky_pipeline, Xte_train, yte_train, cv=cv_tc, scoring='roc_auc')

print("Intern's leaky pipeline (target encoding fit on the FULL data):")
print(f'  Fold AUC: {np.round(cv_leaky, 4)}   mean {cv_leaky.mean():.4f}, SD {cv_leaky.std(ddof=1):.4f}  ← inflated')
print('Leak-free baseline from Section 2.2 (OHE only, no target encoding):')
print(f'  Fold AUC: {np.round(cv_scores_tc, 4)}   mean {mean_tc:.4f}, SD {sd_tc:.4f}  ← realistic')

# --- The size of the lie, measured the course's way: paired, on the same folds ---
d_leak, gain_leak, (lo_leak, hi_leak), verdict_leak = compare_paired(cv_leaky, cv_scores_tc, DELTA_AUC)
print(f'\nLeaky − leak-free, per fold: {np.round(d_leak, 4)}')
print(f'Mean inflation {gain_leak:+.4f} ROC-AUC, interval corrected for fold overlap [{lo_leak:+.4f}, {hi_leak:+.4f}]')
print(f'Read as a comparison at δ = {DELTA_AUC}: {verdict_leak}')

# --- Picture: the same five folds, leaky vs leak-free, joined fold by fold ---
fig, ax = plt.subplots(figsize=(9, 5))
for i in range(5):
    ax.plot([0, 1], [cv_scores_tc[i], cv_leaky[i]], color='gray', alpha=0.7)
ax.scatter(np.zeros(5), cv_scores_tc, s=80, color='steelblue', edgecolor='black', zorder=3, label='Leak-free baseline (OHE)')
ax.scatter(np.ones(5), cv_leaky, s=80, color='crimson', edgecolor='black', zorder=3, label='Leaky (target encoding on full data)')
ax.set_xticks([0, 1])
ax.set_xticklabels(['Leak-free baseline', 'Leaky pipeline'])
ax.set_xlim(-0.4, 1.4)
ax.set_ylabel('5-fold CV ROC-AUC')
ax.set_title('TechCorp — the same five folds, leaky vs leak-free', fontsize=12, fontweight='bold')
ax.legend(loc='lower center')
plt.tight_layout()
plt.show()
print("\nFix: drop the target-encoded columns; fall back to OneHotEncoder(handle_unknown='ignore') inside the Pipeline.")

**Reading the output:**

Two side-by-side CV runs on the same TechCorp dataset, and the numbers are doing exactly what the leak theory predicts.

**The leaky run.** Mean ROC-AUC lands at **0.8057**. The intern's celebration is not unfounded from their perspective — compared to the Section 2.2 baseline of **0.7663**, the leaky pipeline posted a lift of about 4 points, which would be an enormous improvement if it were real.

**The leak-free Section 2.2 baseline.** Mean ROC-AUC **0.7663**. This is the honest number — what the model would actually produce on a new TechCorp employee in production, because the pipeline only saw each fold's training portion during its scaling and encoding fits.

**The gap between them is the *size of the lie*.** Fold by fold the leaky run is ahead by **+0.0450, +0.0343, +0.0506, +0.0328, +0.0348** — a mean of **+0.0395** ROC-AUC, with an interval corrected for fold overlap of **[+0.0248, +0.0542]**. Read as a comparison against the CHRO's 0.01 margin, `compare_paired` prints **"A is better by a worthwhile amount"** — the rule would promote the leaky pipeline without hesitation. That is the real danger: the decision rule judges the numbers it is given, and a leak hands it fiction. A 4-point AUC gap means the leaky model *looks* much better at ranking leavers above stayers; if the intern shipped it, the CHRO would budget retention conversations around that ranking quality and watch the model underperform by that same gap in production. That is how leakage kills careers.

### The fix

**Delete the target-encoded columns and fall back to `OneHotEncoder(handle_unknown='ignore')` inside the `Pipeline`** — exactly what the Section 2.2 baseline already does. OHE never touches `y`, so it is *leak-safe by construction*. The "real" fix — target encoding that refits its per-category means on every CV fold's training portion only — requires a custom transformer (the library `category_encoders` has one called `TargetEncoder` that integrates with sklearn's CV correctly). That is a topic for later in your career; for this course, OHE is the safe default.

### The rule to internalize right now

**If a feature is computed using any row's target, the only safe way to include it is a transformer that refits per CV fold.** `OneHotEncoder` never reads `y`, so it is leak-safe. Target encoders *do* read `y`, so they must live inside the Pipeline and refit per fold. The intern's version failed because the encoding computation ran once on the full DataFrame — outside the fold loop, outside the Pipeline, before the split.

> **A question that often comes up here:** *"If target encoding is so dangerous, why do Kaggle winners use it?"* They use a version that refits per fold — properly implemented, it can give a real boost on high-cardinality categoricals. The dangerous version is the one you just saw (`groupby.transform('mean')` on the full DataFrame), which is how target encoding is often demo'd in quick tutorials and how many of you will first encounter it. The difference between "Kaggle winner move" and "public-leaderboard disaster" is whether the encoder lives inside the `Pipeline` that `cross_val_score` or `GridSearchCV` evaluates. This is not a small distinction; it is the entire game.

---

### 2.4 `FunctionTransformer` — domain features without leakage

Stakeholders almost always want a feature that is not in the raw data. At TechCorp, the HR Business Partner (HRBP — the person who actually talks to employees and managers) is convinced that *ratios matter more than absolute values*: an employee whose `salary_pct_of_market` is 85 is more attrition-prone than one at 130 (they know they are underpaid), and an engineer with 20 manager interactions across 3 projects gets more attention per unit of work than one with 5 interactions across 10 projects.

The HRBP's specific ask: add an `interactions_per_project` ratio defined as `manager_interactions_last_quarter / (projects_completed_last_year + 1)`. (The `+ 1` is a "Laplace smoothing" hack to avoid dividing by zero for employees who completed no projects — a small but real gotcha in any ratio feature.)

### The amateur move vs. the professional move

**The amateur move:** compute the ratio in a pandas `assign` call before splitting, treat the new column as raw data, and feed it to the existing pipeline. On this specific ratio — a pure arithmetic combination of two raw columns, with no dependence on any training statistic — the amateur move actually works, because there is nothing to leak. No mean, no SD, no group-level statistic. A simple function of two existing columns.

**The professional move:** use `FunctionTransformer` to wrap the ratio computation as a pipeline step anyway. This makes the ratio compute fold-by-fold inside the CV loop, and more importantly, it *establishes the habit*: every feature-engineering step is a pipeline step, no exceptions. The habit matters because the moment you add a feature that depends on a training statistic — say, `salary_pct_of_dept_median` (each employee's salary as a fraction of their department's median) — the "compute it in pandas before splitting" approach silently leaks: the department median would be computed using held-out and test rows.

The one-line rule: **if you are even slightly unsure whether a feature engineering step leaks, put it inside the pipeline**. `FunctionTransformer` exists so that rule costs you nothing.

### How it works mechanically

`FunctionTransformer(func, validate=False)` wraps an arbitrary Python function `func` into a sklearn transformer that can be slotted into a `Pipeline`. On every CV fold's training portion, sklearn calls `func(X_train_fold)` to transform the training data; on the held-out portion, it calls `func(X_heldout_fold)`. If `func` is a pure function (no training statistics, no global state), both calls produce the same transformation and nothing can leak. If `func` were to depend on a statistic of its input, sklearn would recompute that statistic per fold — which is exactly the leak-free property you want.

**The test cell below** adds the HRBP's `interactions_per_project` ratio via `FunctionTransformer` and reruns 5-fold CV on the *same* folds as the Section 2.2 baseline. Same decision rule as every comparison today: **`compare_paired(with_ratio, without_ratio, DELTA_AUC)`** — the pipeline with the ratio is the challenger (A), the Section 2.2 baseline is the current champion (B), and the CHRO's base margin of 0.01 ROC-AUC decides. Only *"A is better by a worthwhile amount"* adds the ratio to the champion. Practically equivalent or unresolved: the HRBP's hypothesis is plausible but not shown to matter on this sample, and the simpler pipeline (no ratio) stays.

> 💡 **Gemini Prompt:** "Write a function `add_interactions_per_project(X)` that takes a pandas DataFrame with `manager_interactions_last_quarter` and `projects_completed_last_year`, returns a copy with an added column `interactions_per_project = manager_interactions_last_quarter / (projects_completed_last_year + 1)`. Wrap it in a `FunctionTransformer(validate=False)`. Insert it as the first step in the pipeline (before the `ColumnTransformer`, whose numeric branch now includes the new column) and re-run 5-fold stratified CV on `X_tc_train` with the same `cv_tc`. Print the fold AUCs, mean, and SD of the +ratio pipeline and of the Section 2.2 baseline. Then call `compare_paired(cv_tc_ratio, cv_scores_tc, DELTA_AUC)` — the +ratio pipeline is A, the baseline is B — print the per-fold gains, mean gain, interval, and outcome, and call `show_band(cv_tc_ratio, cv_scores_tc, DELTAS_AUC, 'with the ratio', 'without the ratio')`. Finally draw the mean gain as a dot with a horizontal error bar for the interval against a shaded band from `-DELTA_AUC` to `+DELTA_AUC`."

In [ ]:
# --- FunctionTransformer to add interactions-per-project ratio ---
def add_interactions_per_project(X):
    X = X.copy()
    X['interactions_per_project'] = (
        X['manager_interactions_last_quarter'] /
        (X['projects_completed_last_year'] + 1)
    )
    return X

add_ratio = FunctionTransformer(add_interactions_per_project, validate=False)

NUM_COLS_RATIO = NUM_COLS + ['interactions_per_project']
preprocess_ratio = ColumnTransformer([
    ('num', StandardScaler(), NUM_COLS_RATIO),
    ('cat', OneHotEncoder(handle_unknown='ignore'), CAT_COLS)
])
tc_pipeline_ratio = Pipeline([
    ('add_ratio', add_ratio),
    ('prep', preprocess_ratio),
    ('clf', LogisticRegression(random_state=RANDOM_SEED, max_iter=5000))
])
cv_tc_ratio = cross_val_score(tc_pipeline_ratio, X_tc_train, y_tc_train, cv=cv_tc, scoring='roc_auc')

print('TechCorp pipeline + FunctionTransformer(interactions_per_project)')
print(f'  Fold AUC: {np.round(cv_tc_ratio, 4)}   mean {cv_tc_ratio.mean():.4f}, SD {cv_tc_ratio.std(ddof=1):.4f}')
print('Section 2.2 baseline (no ratio) — the current champion')
print(f'  Fold AUC: {np.round(cv_scores_tc, 4)}   mean {mean_tc:.4f}, SD {sd_tc:.4f}')

# --- The decision: with-ratio (A, challenger) vs without (B, champion), the CHRO's margins ---
d_fe, gain_fe, (lo_fe, hi_fe), verdict_fe = compare_paired(cv_tc_ratio, cv_scores_tc, DELTA_AUC)
print(f'\nGain per fold (+ratio − baseline): {np.round(d_fe, 4)}')
print(f'Mean gain {gain_fe:+.4f} ROC-AUC, interval corrected for fold overlap [{lo_fe:+.4f}, {hi_fe:+.4f}]')
print(f'Outcome at δ = {DELTA_AUC}: {verdict_fe}')
show_band(cv_tc_ratio, cv_scores_tc, DELTAS_AUC, 'with the ratio', 'without the ratio')

# --- Picture: the gain and its interval against ±δ ---
fig, ax = plt.subplots(figsize=(9, 3))
ax.axvspan(-DELTA_AUC, DELTA_AUC, color='gray', alpha=0.15, label=f'±δ (δ = {DELTA_AUC}, declared before scoring)')
ax.axvline(0, color='black', linewidth=0.8)
ax.errorbar(gain_fe, 0, xerr=[[gain_fe - lo_fe], [hi_fe - gain_fe]], fmt='o', color='mediumseagreen',
            capsize=8, markersize=9)
ax.set_yticks([0]); ax.set_yticklabels(['+ratio − baseline'])
ax.set_xlim(-3 * DELTA_AUC, 3 * DELTA_AUC)
ax.set_xlabel('Mean gain in ROC-AUC, with the interval corrected for fold overlap')
ax.set_title(f'TechCorp — does the HRBP ratio earn its place? {verdict_fe}', fontsize=11, fontweight='bold')
ax.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

**Reading the output:**

Two pipelines, five shared folds, and one column of gains. The question is the same as every comparison from nb08 forward: *does the engineered ratio earn a worthwhile gain over the current champion?* — answered by `compare_paired`, not by two point estimates and not by counting folds.

With the ratio the pipeline scores a mean of **0.7685** against the baseline's **0.7663**. Fold by fold the gains are **+0.0057, −0.0036, +0.0051, +0.0034, +0.0003** — a mean gain of **+0.0022** ROC-AUC with an interval corrected for fold overlap of **[−0.0050, +0.0094]**. The whole interval sits inside $\pm 0.01$, so at the CHRO's base margin the outcome is **practically equivalent**: the evidence rules out a lift the CHRO said would matter. The band adds a nuance worth reporting: at the small margin (0.005) the interval pokes past $\pm 0.005$ on both sides, so the outcome there is **unresolved**, and at the large margin it is practically equivalent again. In every case the ratio does not earn its place, so the champion is the **pipeline without the ratio** — robust across the three declared margins.

The sentence for the HRBP: *"Your ratio is a reasonable idea, but on this sample it does not move the ranking of leavers by an amount the CHRO said would matter, so we keep the simpler pipeline. If HR starts collecting a richer measure of manager attention, we will test it the same way."*

The pedagogical point is not whether the ratio helps on *this* synthetic dataset; it is the discipline: **every feature addition is a challenger, tested on the same folds with `compare_paired` against a margin declared in advance**, the same way nb08 tested Ridge against OLS and `C = 0.01` against `C = 1.0`. `FunctionTransformer` lets you turn a stakeholder's hypothesis into a testable feature without leaking.

> **A question that often comes up here:** *"If `FunctionTransformer` doesn't learn anything from the data, why does it need to be inside the `Pipeline`?"* On a strictly static ratio like `manager_interactions / (projects + 1)`, it does not strictly need to be. But the *habit* matters: as soon as you add a feature that depends on a training statistic (a training-set mean, a training-set quantile, a training-set category frequency), the outside-the-pipeline version would silently leak. The professional default is *"every feature-engineering step is a pipeline step,"* without exceptions. That single rule makes it impossible to forget a leak on a day when you are tired or in a rush.

---

## 📝 PAUSE-AND-DO Exercise 2 (10 minutes)

**Task:** Find and fix a second leak — this one a different pattern from Section 2.3's target encoding.

Section 2.3 walked you through a dramatic leak (target encoding on the full dataset). This exercise shows you a subtler one: a feature selector applied *before* the split instead of inside the pipeline. The inflation on this data will be smaller than the target-encoding case — a few thousandths to a few hundredths of a point rather than 0.03 — but the principle is identical and the fix is identical. Small leaks matter too: a 0.005 inflation is half of the 0.01 ROC-AUC margin this course compares models against, which can be the entire reason a `compare_paired` interval appears to clear the margin — and that changes which model ships.

Another teammate sends you the snippet below. They have heard about leakage and are careful — they are **not** doing target encoding. Instead they are using a feature selector to reduce the dimensionality of the one-hot-encoded feature matrix (pruning down from 100+ OHE columns to only the ones that correlate with attrition). Their CV ROC-AUC came out a bit higher than your leak-free baseline. Read the code carefully before running.

```python
# Teammate's pipeline with univariate feature selection
X_all_dummies = pd.get_dummies(techcorp.drop(columns=['left_within_6mo']))
y_all = techcorp['left_within_6mo']

# Select the top-8 features by univariate correlation with the target — on ALL rows
selector = SelectKBest(f_classif, k=8)
X_selected = selector.fit_transform(X_all_dummies, y_all)

# Now split and CV on the pre-selected matrix
Xs_train, Xs_test, ys_train, ys_test = train_test_split(
    X_selected, y_all, test_size=0.20, random_state=RANDOM_SEED, stratify=y_all
)

pipe_sel = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(random_state=RANDOM_SEED, max_iter=5000))
])
cv_sel = cross_val_score(pipe_sel, Xs_train, ys_train, cv=cv_tc, scoring='roc_auc')
print(f'Pre-selected mean AUC: {cv_sel.mean():.4f}')
```

**Your job:**

1. Run the teammate's code and record the leaky mean.
2. Identify the leak in one sentence. *Hint:* `SelectKBest.fit(X, y)` reads `y` — does it do so on training rows only, or on every row?
3. Write a leak-free version by putting `SelectKBest` **inside** a `Pipeline` (chain it after the `ColumnTransformer` from Section 2.2, before the classifier). Run 5-fold stratified CV on `X_tc_train`, `y_tc_train`.
4. Print a final line: `Leak inflated mean ROC-AUC by +<delta>`.
5. Note: even if the inflation on this synthetic data is small, the *principle* is identical to Section 2.3. The Kaggle leaderboard does not reward "small" leaks; it punishes every leak equally when the private test scores reveal.

> **A question that often comes up here:** *"If the leak inflation is only 0.005, does it really matter?"* For a decision a stakeholder is about to make — yes, because the course's comparison margins sit at 0.01, so a 0.005 inflation hands one side half the margin for free and can be the entire reason a `compare_paired` interval clears it on a comparison you care about. More importantly: leaks you tolerate on small data scale catastrophically on big data. The pattern of *"fit outside the pipeline"* is always wrong. Fixing it when the stakes are low (here, 10 minutes in a classroom) is cheaper than discovering the same pattern on a Kaggle private leaderboard at midnight.

---

> 💡 **Gemini Prompt:** "First, reproduce the teammate's leaky pipeline exactly as written: one-hot-encode the full TechCorp dataset with `pd.get_dummies(techcorp.drop(columns=['left_within_6mo']))`, fit `SelectKBest(f_classif, k=8)` on the entire matrix and target, then split the resulting selected matrix once with `train_test_split(..., test_size=0.20, random_state=RANDOM_SEED, stratify=y)`. Build a `Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(random_state=RANDOM_SEED, max_iter=5000))])`, run `cross_val_score` on the training portion with `cv=cv_tc` and `scoring='roc_auc'`, and store `mean_leaky = scores.mean()`. Then build a leak-free pipeline that puts every data-derived step *inside* a single `Pipeline`: `ColumnTransformer([('num', StandardScaler(), NUM_COLS), ('cat', OneHotEncoder(handle_unknown='ignore'), CAT_COLS)])` → `SelectKBest(f_classif, k=8)` → `LogisticRegression(random_state=RANDOM_SEED, max_iter=5000)`. Run `cross_val_score` on `X_tc_train`, `y_tc_train` with `cv=cv_tc` and `scoring='roc_auc'` and store `mean_fixed = scores.mean()`. Print both means with 4-decimal precision and the delta `mean_leaky - mean_fixed`. End with a single printed sentence naming where the leak is in the teammate's code."
>
> **After running, verify:**
> - The leaky-pipeline mean ROC-AUC prints first (slightly higher)
> - The leak-free mean ROC-AUC prints next (slightly lower — typically by 0.005 to 0.01 on this data)
> - The delta `mean_leaky - mean_fixed` is positive and equals the inflation the leak buys
> - The final printed sentence identifies the leak as `SelectKBest.fit(X, y)` running on the full dataset *before* the split
> - All numerical outputs use standard decimal format — no scientific notation

In [ ]:
# YOUR SOLUTION CODE HERE
# Hint: Use the Gemini prompt above for step-by-step guidance.
#
# What to do:
#   1. Reproduce the teammate's pre-select-then-split pipeline and record cv_sel.mean()
#   2. Build a leak-free pipeline: ColumnTransformer (Section 2.2) → SelectKBest(k=8) → LogisticRegression
#   3. Run cross_val_score on X_tc_train, y_tc_train with cv=cv_tc and scoring='roc_auc'
#   4. Print both means and the delta
#   5. In the analysis cell below, explain WHERE the leak is (one sentence)


### YOUR ANALYSIS:

**Question 1:** In one sentence, where is the leak in the teammate's code?  
[Your answer]

**Question 2:** By how much did the leak inflate the mean ROC-AUC? Is that inflation large enough to push a `compare_paired` interval with a rival model past a margin it should not have cleared?  
[Your answer]

**Question 3:** What is the one-line rule you would teach this teammate so this never happens again?  
[Your answer]

---

## 3. Section C — Toolkit Recap: What You Hold After nb01–nb09

This section is your one-page reference for the full mid-course toolkit. Skim it now; treat it as the cheat sheet for any new business case you encounter (*"what target? what metric? what split? where is the leakage risk?"*). Nothing below is new — every item lives in a notebook you have already worked through. The point of the recap is to lay the toolkit out in one place so future cases hit a prepared mind, not a scattered one.

### 3.1 Concepts — the mental models the toolkit rests on

The course's core *concepts* are not sklearn methods; they are the language any business-case prompt uses. Each is introduced explicitly in a specific notebook and is part of the toolkit you will reach for from here on.

- **The statistical-learning frame ($Y = f(X) + \epsilon$, nb01).** Every problem you have seen reduces to estimating $f$. Two flavors: **regression** (continuous $Y$, like house prices in nb01–nb05) and **classification** (categorical $Y$, like cancer or attrition in nb06–nb09). Knowing which flavor a business case calls for is usually the first decision you have to make.
- **Bias–variance tradeoff (nb01, nb04, nb05).** Bias is how wrong the model is on average; variance is how much its predictions move when the training sample changes. High bias → underfitting; high variance → overfitting. Regularization (nb05) is *literally* a knob that buys lower variance at the cost of a sliver of bias.
- **Overfitting vs underfitting (nb04).** Diagnose by comparing training scores with held-out scores (a validation split in nb04, cross-validation from nb08 on): train ≫ held-out = overfitting; both equally bad = underfitting. nb04's polynomial blow-up was the canonical example; nb05's Ridge/Lasso was the canonical fix.
- **Curse of dimensionality (nb01, nb04).** As the feature count grows, every training point ends up alone in a sparsely populated neighborhood and naive distance-based reasoning falls apart. This is *why* regularization matters as soon as you add interactions or polynomials.
- **Data leakage (nb01, nb02, nb09).** Two flavors. **Target leakage**: a feature is computed using information that would not be available at prediction time (the target's own value, future timestamps, post-outcome columns). **Train–test contamination**: any `.fit(...)` step that touches held-out data — scaling, encoding, target encoding, feature selection — *before* the split. nb09's two case studies (target encoding on the full dataset and `SelectKBest` outside the pipeline) are the canonical traps you should be able to spot in code.

### 3.2 Workflow — the four moves of the inner cycle

These four moves are **the inner cycle of nb01's End-to-End Predictive Modeling Workflow**, specifically Steps 4–7 (Baseline → Advanced Models → Comparison → Selection). nb01's full 10-step framework still owns the project lifecycle (Setup → Data Loading → EDA → Data Prep → Baseline → Advanced Models → Comparison → Selection → Final Training → Test Predictions → Reporting). The four moves below are how *one model evaluation* fits inside that framework — a full project executes the inner cycle once per candidate model family on the roster, then settles at nb01 Step 8 (Final Training) and opens the test set at nb01 Step 9 (the nb14 ceremony).

You did not see all four moves at once until nb08. nb01–nb07 set up the first three (EDA + split / Pipeline / Honest evaluation); nb08 added `compare_paired`, which locks the cycle; nb09 scaled it across grids of candidates.

1. **EDA + split (nb01, nb06, nb08).** Audit dtypes, missingness, target distribution; then split. nb01–nb07 taught the train/validation/test split; from nb09 on, split once — `train_test_split(X, y, test_size=0.20, random_state=RANDOM_SEED)`, stratified for classification — and do all model work by 5-fold CV on `X_train`. The test set is locked until nb14. *Maps to nb01 Steps 1–3.*
2. **Build the pipeline (nb02, nb09).** `ColumnTransformer` for column-type-specific preprocessing — numeric → `StandardScaler`; categorical → `OneHotEncoder(handle_unknown='ignore')` — then your model, all wrapped in a single `Pipeline`. Every data-derived step lives *inside* the pipeline so CV refits each fold cleanly. *Maps to nb01 Step 3.*
3. **Evaluate honestly (nb03, nb07, nb08).** Pick a metric the business cares about (MAE / RMSE / $R^2$ for regression; precision / recall / F1 / ROC-AUC for classification, with thresholding when costs are asymmetric). Run 5-fold CV on `X_train`, never on `X_test`. Report each model's five fold scores, their mean, and their SD — no interval around a single model's score, because the folds share training rows. *Maps to nb01 Steps 4 (first pass) and 6 (subsequent passes).*
4. **Compare with `compare_paired` against a predeclared margin (nb08, nb09).** Declare the margins before scoring. Score the challenger and the current champion on the *identical* folds and call `compare_paired(challenger, champion, δ)`: the interval around the mean per-fold gain, corrected for fold overlap, gives one of four outcomes — A better by a worthwhile amount (promote), B better by a worthwhile amount, practically equivalent, or unresolved (keep the champion in all three). A search's best cell is a nominee that faces the current setting the same way. *This decision primitive is the spine of model selection from here on. Maps to nb01 Step 7.*

### 3.3 Tools — the sklearn primitives at your fingertips

| Layer | Primitive | Where it lives in the course |
|---|---|---|
| Preprocessing | `StandardScaler`, `OneHotEncoder(handle_unknown='ignore')`, `FunctionTransformer` | nb02, nb09 |
| Composition | `Pipeline`, `ColumnTransformer` | nb02, nb09 |
| Regression | `LinearRegression`, `Ridge`, `Lasso` | nb03–nb05 |
| Classification | `LogisticRegression` (`C` is regularization, `class_weight='balanced'` for imbalance) | nb06, nb07 |
| Splitting | `train_test_split(stratify=y)`, `KFold`, `StratifiedKFold` | nb01, nb06, nb08 |
| Evaluation | `cross_val_score`, `cross_val_predict`, regression metrics (`mean_absolute_error`, `mean_squared_error`, `r2_score`), classification metrics (`roc_auc_score`, `classification_report`, `confusion_matrix`) | nb03, nb07, nb08 |
| Tuning | `GridSearchCV`, `RandomizedSearchCV` (read `cv_results_`; the best cell is a nominee that faces the current setting with `compare_paired`) | nb09 |

### 3.4 Decision rules — the heuristics you now apply

These are the one-line rules you should have memorized walking into any new analysis. They apply to every business case from here on.

- **Scale when the model uses distances or coefficients.** Logistic regression, Ridge, Lasso → scale. Trees and tree ensembles (nb11–nb13) → no scaling needed.
- **Stratify when classes are imbalanced.** Classification splits should be stratified, and so should `KFold` (use `StratifiedKFold`).
- **Metric choice flows from cost asymmetry.** If false negatives are expensive and false positives are cheap, optimize recall; if the opposite, optimize precision. F1 balances both. ROC-AUC is metric-agnostic and good for ranking; PR-AUC is better when the positive class is rare. A typical business case (*"missed fraud costs USD 50,000; a false alarm costs USD 50"* — pick the metric) makes the cost asymmetry concrete.
- **Ridge keeps everything small; Lasso zeros things out.** Ridge for stable coefficient estimates with all features kept; Lasso for automatic feature selection (sparsity).
- **`C` is the *inverse* of regularization strength** (LogReg). Small `C` (e.g., 0.01) → strong regularization → simpler model. Large `C` (e.g., 100) → weak regularization → flexible boundary. `C = 1.0` is sklearn's default.
- **`compare_paired` (the course's spine).** Declare the margin band before scoring (base decides). Score the challenger (A) and the current champion (B) on identical folds; the interval around the mean per-fold gain, corrected for fold overlap, is read against ±δ: above +δ → the challenger is better by a worthwhile amount and takes the job; below −δ → the champion is better by a worthwhile amount; inside ±δ → practically equivalent; anything else → unresolved. In the last three cases the champion stays. Fold wins are description, never the verdict. Report every model's fold scores, mean, and SD — never a point estimate without its spread.
- **Leakage rule (one line).** Any `.fit(...)` call on data — scaler, encoder, imputer, selector, target encoder — must live *inside* the `Pipeline` that `cross_val_score` or `GridSearchCV` evaluates. If a transform takes $X$ or $y$ during `fit` and lives outside, it is leaking.

> **A question that often comes up here:** *"How do I make sure I can apply this toolkit on a new case?"* Re-run nb01 → nb09 in order, paying attention to *why* each notebook exists, not just *what* code it runs. Any new business case hands you brand-new context and asks you to choose target / metric / split / pipeline / risk-flag — none of those decisions are looked up in code; they all come from the concepts and decision rules above. If you can write one paragraph for each row of the table in 3.3 explaining when you would reach for that primitive, you can wield the toolkit on whatever comes next.

---

## 4. Wrap-Up: Key Takeaways

### What We Learned Today:

1. **`GridSearchCV` is nb08's CV ritual, run on every point in a grid.** Every row of `cv_results_` carries five fold scores, a mean, and an SD. The table describes; it does not decide.
2. **A search's best cell is a nominee, not a champion.** It was picked for scoring highest on these folds, so its mean is optimistic. It faces the current setting — OLS for HomeValue, the default `C = 1.0` for MedScreen — with `compare_paired`, and it takes the job only on a worthwhile gain. Today neither nominee did: both came out practically equivalent, robustly across the band.
3. **`RandomizedSearchCV` is the same idea for larger grids.** Sample `n_iter` points from distributions (`loguniform`, `uniform`) when an exhaustive grid would take too long. The output format is identical, and its best row is a nominee too.
4. **`ColumnTransformer` is the bridge to real-world data.** Numeric columns need `StandardScaler`; categorical columns need `OneHotEncoder(handle_unknown='ignore')`. Both branches live inside one `ColumnTransformer`, which lives inside one `Pipeline`. That pattern is the template for every final-project pipeline from here on.
5. **`FunctionTransformer` embeds domain features without leaking — and every new feature is a challenger.** Test it with `compare_paired` on the same folds against the margin declared in advance; the HRBP's ratio came out practically equivalent at the CHRO's base margin, so the simpler pipeline stays.
6. **Leakage hides inside `fit` calls on the full dataset, and the decision rule cannot see it.** Any step that takes `X` (for scaling, selection, encoding) or `y` (for feature selection, target encoding) must live inside the `Pipeline`. Today's target-encoding leak was large enough that `compare_paired` would have promoted the leaky pipeline — the rule judges the numbers it is given, so the pipeline has to be honest first.

### Critical Rules:

> **"If a preprocessing step uses any data statistic, it belongs inside the `Pipeline`."**

> **"`cv_results_` is a stack of nb08 runs, and its top row is a nominee."**

> **"Same folds, `compare_paired`, read against δ: only a worthwhile gain promotes the more complex model."**

### Next Steps:

- **Next on the calendar: the in-class midterm, then nb11 (decision trees).** Every new model class from nb11 to nb13 introduces more hyperparameters — `max_depth`, `min_samples_leaf`, `n_estimators`, `learning_rate` — and today's reflex is how you will handle them: search, treat the best cell as a nominee, and let it face the current champion with `compare_paired`.
- **Your final project and the Course Case Competition** use the same pipeline discipline — `ColumnTransformer → FunctionTransformer → GridSearchCV → nominee → compare_paired against the current champion → refit on all training data`. The competition is a group S&P 500 forecasting competition on Kaggle that starts with nb18; Section 2's leakage case studies are there so you do not ship a *leaky* submission or a leaky poster.

---

## Participation Assignment Submission Instructions

### To Submit This Notebook:

1. **Complete all exercises**: Fill in your code and analysis in both PAUSE-AND-DOs.
2. **Verify outputs**: Restart the runtime (`Runtime → Restart session`) and run all cells top to bottom. Every cell should execute without error.
3. **Download**: `File → Download → Download .ipynb` to save your completed notebook.
4. **Submit on Brightspace** in the Participation Assignments section.

### Grading:

- Exercise 1 complete with the `compare_paired` outcome and the champion `C`: 50%
- Exercise 2 complete with leak diagnosis: 50%

---

## 5. Bibliography

- James, G., Witten, D., Hastie, T., & Tibshirani, R. (2023). *An Introduction to Statistical Learning with Python* (ISLP), Ch. 5 (Resampling Methods) and Ch. 6 (Linear Model Selection and Regularization). Springer.
- Provost, F., & Fawcett, T. (2013). *Data Science for Business*. Chapters on overfitting, evaluation, and pipeline discipline.
- scikit-learn User Guide — [Tuning the hyper-parameters of an estimator](https://scikit-learn.org/stable/modules/grid_search.html)
- scikit-learn User Guide — [ColumnTransformer](https://scikit-learn.org/stable/modules/compose.html#columntransformer-for-heterogeneous-data)
- scikit-learn User Guide — [Common pitfalls in interpretation and evaluation](https://scikit-learn.org/stable/common_pitfalls.html)
- Kaufman, S., Rosset, S., & Perlich, C. (2012). *Leakage in Data Mining: Formulation, Detection, and Avoidance.*
- Nadeau, C., & Bengio, Y. (2003). Inference for the generalization error. *Machine Learning*, 52, 239–281 — the correction for fold overlap in `compare_paired`.
- Cawley, G. C., & Talbot, N. L. C. (2010). On over-fitting in model selection and subsequent selection bias in performance evaluation. *Journal of Machine Learning Research*, 11, 2079–2107 — why a search's best cell is optimistic.

---

<center>

Thank you!

</center>